# Day 24 — SQL for Data Science — Foundations
## Nepal Bank Transaction Analysis

**Objective:** Master fundamental SQL concepts through hands-on practice with a Nepal bank transaction dataset

---

## 1. Environment Setup 

We'll use SQLite3 which comes built-in with Python's standard library.

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta
import random

# Create in-memory database for this session
conn = sqlite3.connect('banks.db')

cursor = conn.cursor()

print("SQLite database ready!")
print(f"SQLite version: {sqlite3.sqlite_version}")

A cursor is an object used to execute SQL commands.

## 2. Relational DB Concepts 

### Key Concepts:
- **Table**: Structured data with rows (records) and columns (attributes)
- **Primary Key (PK)**: Unique identifier for each row
- **Foreign Key (FK)**: References primary key of another table
- **Schema**: Blueprint of database structure

### Our Database Schema:

**Table 1: customers**
- customer_id (PK) - Unique customer identifier
- full_name - Customer's full name
- province - Province in Nepal
- district - District in Nepal
- account_type - Savings/Current
- created_date - Account opening date

**Table 2: transactions**
- transaction_id (PK) - Unique transaction identifier
- customer_id (FK) - References customers.customer_id
- transaction_date - Date of transaction
- transaction_type - Deposit/Withdrawal/Transfer
- amount - Transaction amount in NPR
- branch - Bank branch location
- description - Transaction description

In [ ]:
# Create customers table
cursor.execute('''
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    full_name TEXT NOT NULL,
    province TEXT,
    district TEXT,
    account_type TEXT,
    created_date DATE
)
''')

In [ ]:
# Create transactions table
cursor.execute('''
CREATE TABLE transactions (
    transaction_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    transaction_date DATE,
    transaction_type TEXT,
    amount REAL,
    branch TEXT,
    description TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
)
''')

print("Tables created successfully!")
print("Schema:")
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
for table in cursor.fetchall():
    print(f"  - {table[0]}")

## 3. Sample Data Generation 

Let's populate our database with realistic Nepal bank data.

In [ ]:
# Generate customer data
customers_data = [
    (1, 'Rajesh Sharma', 'Bagmati', 'Kathmandu', 'Savings', '2024-01-15'),
    (2, 'Sita Thapa', 'Gandaki', 'Pokhara', 'Current', '2024-02-01'),
    (3, 'Krishna Gurung', 'Lumbini', 'Butwal', 'Savings', '2024-02-20'),
    (4, 'Sunita Rai', 'Koshi', 'Biratnagar', 'Savings', '2024-03-05'),
    (5, 'Hari Shrestha', 'Bagmati', 'Lalitpur', 'Current', '2024-03-15'),
    (6, 'Gita Adhikari', 'Sudurpashchim', 'Dhangadhi', 'Savings', '2024-04-01'),
    (7, 'Ram Poudel', 'Karnali', 'Surkhet', 'Savings', '2024-04-10'),
    (8, 'Maya Karki', 'Bagmati', 'Bhaktapur', 'Current', '2024-05-01'),
    (9, 'Bikram Shah', 'Gandaki', 'Lekhnath', 'Savings', '2024-05-15'),
    (10, 'Laxmi Tamang', 'Province 1', 'Dharan', 'Savings', '2024-06-01')
]

cursor.executemany('''
INSERT INTO customers VALUES (?, ?, ?, ?, ?, ?)
''', customers_data)

In [ ]:
# Generate transaction data
transactions_data = [
    (1, 1, '2024-05-20', 'Deposit', 50000, 'Kathmandu Main', 'Salary deposit'),
    (2, 2, '2024-05-21', 'Withdrawal', 15000, 'Pokhara Branch', 'Cash withdrawal'),
    (3, 1, '2024-05-22', 'Transfer', 20000, 'Kathmandu Main', 'Transfer to savings'),
    (4, 3, '2024-05-23', 'Deposit', 75000, 'Butwal Branch', 'Business payment'),
    (5, 4, '2024-05-24', 'Withdrawal', 25000, 'Biratnagar Branch', 'ATM withdrawal'),
    (6, 5, '2024-05-25', 'Deposit', 100000, 'Lalitpur Branch', 'Fixed deposit'),
    (7, 2, '2024-05-26', 'Transfer', 5000, 'Pokhara Branch', 'Mobile recharge'),
    (8, 6, '2024-05-27', 'Withdrawal', 30000, 'Dhangadhi Branch', 'Shopping'),
    (9, 7, '2024-05-28', 'Deposit', 45000, 'Surkhet Branch', 'Remittance'),
    (10, 8, '2024-05-29', 'Deposit', 25000, 'Bhaktapur Branch', 'Rent payment'),
    (11, 9, '2024-05-30', 'Withdrawal', 12000, 'Lekhnath Branch', 'Daily expenses'),
    (12, 10, '2024-06-01', 'Deposit', 60000, 'Dharan Branch', 'Bonus deposit'),
    (13, 1, '2024-06-02', 'Deposit', 15000, 'Kathmandu Main', 'Freelance payment'),
    (14, 3, '2024-06-03', 'Withdrawal', 40000, 'Butwal Branch', 'Vendor payment'),
    (15, 5, '2024-06-04', 'Transfer', 30000, 'Lalitpur Branch', 'Investment'),
    (16, 2, '2024-06-05', 'Deposit', 35000, 'Pokhara Branch', 'Salary deposit'),
    (17, 4, '2024-06-06', 'Deposit', 28000, 'Biratnagar Branch', 'Business revenue'),
    (18, 6, '2024-06-07', 'Deposit', 22000, 'Dhangadhi Branch', 'Government allowance'),
    (19, 7, '2024-06-08', 'Withdrawal', 8000, 'Surkhet Branch', 'Grocery shopping'),
    (20, 8, '2024-06-09', 'Withdrawal', 45000, 'Bhaktapur Branch', 'Medical expenses'),
    (21, 9, '2024-06-10', 'Deposit', 32000, 'Lekhnath Branch', 'Consulting fee'),
    (22, 10, '2024-06-11', 'Withdrawal', 18000, 'Dharan Branch', 'Travel expenses'),
    (23, 1, '2024-06-12', 'Withdrawal', 30000, 'Kathmandu Main', 'Utility bills'),
    (24, 3, '2024-06-13', 'Deposit', 50000, 'Butwal Branch', 'Project payment'),
    (25, 5, '2024-06-14', 'Withdrawal', 20000, 'Lalitpur Branch', 'Dining expenses')
]

cursor.executemany('''
INSERT INTO transactions VALUES (?, ?, ?, ?, ?, ?, ?)
''', transactions_data)

conn.commit()

print(" Sample data inserted successfully!")
print(f"Customers: {len(customers_data)}")
print(f"Transactions: {len(transactions_data)}")

## 4. Basic Queries: SELECT, WHERE, ORDER BY, LIMIT, DISTINCT 

### SELECT Statement
Used to retrieve data from tables.

**Syntax:** `SELECT column1, column2 FROM table_name;`

In [ ]:
# Example 1: SELECT all columns
print(" All customers:")
df_customers = pd.read_sql_query("SELECT * FROM customers", conn)
print(df_customers)
print("\n" + "="*80 + "\n")

# Example 2: SELECT specific columns
print(" Customer names and accounts:")
df_select = pd.read_sql_query("SELECT full_name, account_type FROM customers", conn)
print(df_select)

In [ ]:
# WHERE clause: Filtering rows

# Example 1: Filter by province
print(" Customers from Bagmati province:")
df_where = pd.read_sql_query("SELECT full_name, district FROM customers WHERE province = 'Bagmati'", conn)
print(df_where)
print("\n" + "="*80 + "\n")

# Example 2: Filter by amount
print(" Transactions > 50000 NPR:")
df_where2 = pd.read_sql_query("SELECT * FROM transactions WHERE amount > 50000", conn)
print(df_where2)

In [ ]:
# ORDER BY and LIMIT

# Example 1: Order transactions by amount (descending)
print(" Top 5 largest transactions:")
df_order = pd.read_sql_query('''
SELECT transaction_id, amount, branch 
FROM transactions 
ORDER BY amount DESC 
LIMIT 5
''', conn)
print(df_order)
print("\n" + "="*80 + "\n")

# Example 2: Oldest customers
print(" 3 oldest customers:")
df_oldest = pd.read_sql_query('''
SELECT full_name, created_date 
FROM customers 
WHERE account_type = 'Savings'
ORDER BY created_date ASC 
LIMIT 3
''', conn)
print(df_oldest)

In [ ]:
# DISTINCT: Get unique values

print(" Unique branches in Nepal:")
df_distinct = pd.read_sql_query("SELECT DISTINCT branch FROM transactions ORDER BY branch", conn)
print(df_distinct)
print("\n" + "="*80 + "\n")

print("Unique transaction types:")
df_types = pd.read_sql_query("SELECT DISTINCT transaction_type FROM transactions", conn)
print(df_types)

## 5. Aggregations: COUNT, SUM, AVG, MIN, MAX, GROUP BY, HAVING

### Aggregate Functions
- **COUNT**: Number of rows
- **SUM**: Total of numeric column
- **AVG**: Average value
- **MIN**: Minimum value
- **MAX**: Maximum value

### GROUP BY: Groups rows with same values
### HAVING: Filter groups (like WHERE for groups)

In [ ]:
# Basic aggregations
print("Transaction Statistics:")
df_stats = pd.read_sql_query('''
SELECT 
    COUNT(*) as total_transactions,
    SUM(amount) as total_amount_npr,
    AVG(amount) as avg_amount_npr,
    MIN(amount) as min_amount_npr,
    MAX(amount) as max_amount_npr
FROM transactions
''', conn)
print(df_stats)

In [ ]:
# GROUP BY: Analyze by transaction type
print("Transactions by Type:")
df_group = pd.read_sql_query('''
SELECT 
    transaction_type
FROM transactions
GROUP BY transaction_type
''', conn)
print(df_group)
print("\n" + "="*80 + "\n")

In [ ]:
# GROUP BY: Analyze by transaction type
print("Transactions by Type:")
df_group = pd.read_sql_query('''
SELECT 
    transaction_type,
    COUNT(*) as count,
    SUM(amount) as total_npr,
    AVG(amount) as avg_npr,
    MIN(amount) as min_npr,
    MAX(amount) as max_npr
FROM transactions
GROUP BY transaction_type
''', conn)
print(df_group)
print("\n" + "="*80 + "\n")

In [ ]:
# GROUP BY with HAVING: Only groups with conditions
print("Branches with total transactions > 50,000 NPR:")
df_having = pd.read_sql_query('''
SELECT 
    branch,
    COUNT(*) as transaction_count,
    SUM(amount) as total_amount
FROM transactions
GROUP BY branch
HAVING SUM(amount) > 50000
ORDER BY total_amount DESC
''', conn)
print(df_having)

In [ ]:
# Multiple GROUP BY columns
print("Analysis by Province and Account Type:")
df_multi = pd.read_sql_query('''
SELECT 
    c.province,
    c.account_type,
    COUNT(t.transaction_id) as transaction_count,
    SUM(t.amount) as total_amount
FROM customers c
JOIN transactions t ON c.customer_id = t.customer_id
GROUP BY c.province, c.account_type
ORDER BY c.province, total_amount DESC
''', conn)
print(df_multi)

## 6. Filtering with LIKE, IN, BETWEEN, IS NULL

In [ ]:
# LIKE: Pattern matching
print("Customers whose name starts with 'S':")
df_like = pd.read_sql_query("SELECT full_name FROM customers WHERE full_name LIKE 'S%'", conn)
print(df_like)
print("\n" + "="*80 + "\n")

In [ ]:
# LIKE: Pattern matching
print("Customers whose name starts with 'S':")
df_like = pd.read_sql_query("SELECT district FROM customers WHERE district LIKE 'B%'", conn)
print(df_like)
print("\n" + "="*80 + "\n")

In [ ]:
# IN: Match multiple values
print("Customers from Kathmandu or Pokhara:")
df_in = pd.read_sql_query('''
SELECT full_name, district ,province
FROM customers 
WHERE province IN ('Bagmati', 'Gandaki', 'Lumbini')
''', conn)
print(df_in)
print("\n" + "="*80 + "\n")

In [ ]:
# BETWEEN: Range filter
print("Transactions between 20,000 and 50,000 NPR:")
df_between = pd.read_sql_query('''
SELECT transaction_id, amount, branch 
FROM transactions 
WHERE amount BETWEEN 30000 AND 40000 
''', conn)
print(df_between)

In [ ]:
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE customers_between1 AS
SELECT transaction_id, amount, branch
FROM transactions
WHERE amount BETWEEN 30000 AND 40000
''')

conn.commit()

df_between = pd.read_sql_query(
    "SELECT * FROM customers_between",
    conn
)

print(df_between)

## 7. Aliases and Calculated Columns

### Aliases: Rename columns or tables using `AS`
### Calculated Columns: Create new columns from existing data

In [ ]:
# Aliases and calculated columns
print(" Transaction analysis with calculated columns:")
df_calc = pd.read_sql_query('''
SELECT 
    transaction_id,
    amount,
    amount * 0.13 as vat_npr,  -- 13% VAT
    amount * 1.13 as total_with_vat,
    CASE 
        WHEN amount > 50000 THEN 'Large'
        WHEN amount > 20000 THEN 'Medium'
        ELSE 'Small'
    END as transaction_size
FROM transactions
LIMIT 5
''', conn)
print(df_calc)
print("\n" + "="*80 + "\n")

In [ ]:
# Complex calculated columns with functions
print("📊 Date analysis:")
df_date = pd.read_sql_query('''
SELECT 
    transaction_date,
    amount,
    strftime('%Y', transaction_date) as year,
    strftime('%m', transaction_date) as month,
    strftime('%d', transaction_date) as day,
    strftime('%W', transaction_date) as week_number
FROM transactions
ORDER BY transaction_date
LIMIT 5
''', conn)
print(df_date)

## 8. JOIN: Combining Tables (Bonus)

### INNER JOIN: Returns rows where there's a match in both tables

In [ ]:
# Customer transaction analysis with JOIN
print("👥 Customer transaction summary:")
df_join = pd.read_sql_query('''
SELECT 
    c.full_name,
    c.province,
    c.district,
    COUNT(t.transaction_id) as transaction_count,
    SUM(t.amount) as total_transactions,
    AVG(t.amount) as avg_transaction,
    MAX(t.amount) as largest_transaction
FROM customers c
JOIN transactions t ON c.customer_id = t.customer_id
GROUP BY c.customer_id
ORDER BY total_transactions DESC
''', conn)
print(df_join)

# SQL JOIN Types

A **JOIN** is used to combine rows from two or more tables based on a related column.

---

## 1. INNER JOIN

Returns only the rows that have matching values in both tables.

**Syntax:**

```sql
SELECT *
FROM customers c
INNER JOIN transactions t
ON c.customer_id = t.customer_id;
```

**Note:** `JOIN` and `INNER JOIN` mean the same thing.

Your query uses an **INNER JOIN** because:

```sql
FROM customers c
JOIN transactions t
ON c.customer_id = t.customer_id
```

---

## 2. LEFT JOIN (LEFT OUTER JOIN)

Returns all rows from the left table and the matching rows from the right table. If there is no match, the right-side columns contain `NULL`.

**Syntax:**

```sql
SELECT
    c.full_name,
    t.amount
FROM customers c
LEFT JOIN transactions t
ON c.customer_id = t.customer_id;
```

---

## 3. RIGHT JOIN (RIGHT OUTER JOIN)

Returns all rows from the right table and the matching rows from the left table. If there is no match, the left-side columns contain `NULL`.

**Syntax:**

```sql
SELECT
    c.full_name,
    t.amount
FROM customers c
RIGHT JOIN transactions t
ON c.customer_id = t.customer_id;
```

> **Note:** SQLite does **not** support `RIGHT JOIN`.

---

## 4. FULL OUTER JOIN

Returns all rows from both tables. Matching rows are combined, while non-matching rows contain `NULL` values.

**Syntax:**

```sql
SELECT
    c.full_name,
    t.amount
FROM customers c
FULL OUTER JOIN transactions t
ON c.customer_id = t.customer_id;
```

> **Note:** SQLite does **not** support `FULL OUTER JOIN`.

---

## 5. CROSS JOIN

Returns every possible combination of rows from both tables.

**Syntax:**

```sql
SELECT
    c.full_name,
    t.transaction_id
FROM customers c
CROSS JOIN transactions t;
```

If there are:
- 5 customers
- 10 transactions

The result will contain **5 × 10 = 50 rows**.

---



---

# JOIN Summary

| JOIN Type | Description |
|-----------|-------------|
| **INNER JOIN** | Returns only matching rows from both tables. |
| **LEFT JOIN** | Returns all rows from the left table and matching rows from the right table. |
| **RIGHT JOIN** | Returns all rows from the right table and matching rows from the left table. *(Not supported in SQLite)* |
| **FULL OUTER JOIN** | Returns all rows from both tables. *(Not supported in SQLite)* |
| **CROSS JOIN** | Returns every possible combination of rows from both tables. |


---

## SQLite Support

| JOIN Type | Supported in SQLite |
|-----------|---------------------|
| INNER JOIN | ✅ Yes |
| LEFT JOIN | ✅ Yes |
| RIGHT JOIN | ❌ No |
| FULL OUTER JOIN | ❌ No |
| CROSS JOIN | ✅ Yes |


## 9. Lab Exercise

### 🎯 Challenge: Nepal Bank Transaction Analysis

Write SQL queries to answer the following business questions:

In [ ]:
# Challenge 1: Find the top 3 customers by total transaction amount
print("🏆 Top 3 Customers by Transaction Amount:")
q1 = '''
SELECT 
    c.full_name,
    SUM(t.amount) as total_spent
FROM customers c
JOIN transactions t ON c.customer_id = t.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC
LIMIT 5
'''
df_q1 = pd.read_sql_query(q1, conn)
print(df_q1)
print("\n" + "="*80 + "\n")

In [ ]:
# Challenge 2: Calculate average transaction amount by province
print("📊 Average Transaction Amount by Province:")
q2 = '''
SELECT 
    c.province,
    COUNT(t.transaction_id) as num_transactions,
    ROUND(AVG(t.amount), 2) as avg_amount
FROM customers c
JOIN transactions t ON c.customer_id = t.customer_id
GROUP BY c.province
HAVING COUNT(t.transaction_id) >= 2
ORDER BY avg_amount DESC
'''
df_q2 = pd.read_sql_query(q2, conn)
print(df_q2)
print("\n" + "="*80 + "\n")

In [ ]:
# Challenge 3: Find branches with only deposits
print("🏦 Branches with Only Deposit Transactions:")
q3 = '''
SELECT 
    branch,
    COUNT(*) as total_transactions,
    SUM(CASE WHEN transaction_type = 'Deposit' THEN 1 ELSE 0 END) as deposits
FROM transactions
GROUP BY branch
HAVING deposits = COUNT(*)
'''
df_q3 = pd.read_sql_query(q3, conn)
print(df_q3)
print("\n" + "="*80 + "\n")

In [ ]:
# Challenge 4: Monthly transaction trend
print("📈 Monthly Transaction Trend:")
q4 = '''
SELECT 
    strftime('%Y-%m', transaction_date) as month,
    COUNT(*) as transaction_count,
    SUM(amount) as total_amount,
    ROUND(AVG(amount), 2) as avg_amount
FROM transactions
GROUP BY month
ORDER BY month
'''
df_q4 = pd.read_sql_query(q4, conn)
print(df_q4)

## 10. Summary & Key Takeaways

### 🎓 What We Learned Today:

1. **Relational DB Concepts**: Tables, PK/FK relationships, schemas
2. **Basic Queries**: SELECT, WHERE, ORDER BY, LIMIT, DISTINCT
3. **Aggregations**: COUNT, SUM, AVG, MIN, MAX with GROUP BY and HAVING
4. **Filtering**: LIKE, IN, BETWEEN, IS NULL for precise data filtering
5. **Advanced**: Aliases, calculated columns, JOIN operations

### 📌 Real-World Applications:
- **Banking**: Transaction monitoring, customer analytics
- **Business**: Sales analysis, customer segmentation
- **Government**: Policy making, economic trend analysis

### 🔑 Key SQL Tips for Data Science:
- Always use `LIMIT` when exploring large datasets
- Use `EXPLAIN QUERY PLAN` to optimize performance
- Document your queries for reproducibility
- Test with small subsets before running on production data

### 📚 Next Steps:
- Practice with real datasets
- Learn subqueries and window functions
- Explore advanced joins and CTEs
- Connect SQL with Python for data science workflows

In [ ]:
# Close connection
conn.close()
print("✅ Database connection closed")